# 03 LightGBM Training

Train and evaluate LightGBM with 5-fold CV. Then train the final serving model on all available training rows.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

from amex_default.config import PLOTS_DIR, REPORTS_DIR
from amex_default.data import get_feature_columns, load_train_features, split_features_target
from amex_default.train_lightgbm import save_cv_artifacts, save_final_model, train_cv, train_final_model


In [ ]:
df = load_train_features()
X, y = split_features_target(df)
feature_cols = get_feature_columns(df)

print(f"Rows: {len(df):,}")
print(f"Features: {len(feature_cols):,}")
print(y.value_counts(normalize=True))


In [ ]:
result = train_cv(X, y, customer_ids=df["customer_ID"])
save_cv_artifacts(result)

metrics = result["metrics"]
metric_cols = [
    "model", "n_rows", "n_features", "roc_auc", "pr_auc",
    "precision", "recall", "f1", "threshold",
    "training_time_seconds", "inference_time_seconds", "total_time_seconds",
]
display(pd.DataFrame([metrics]).reindex(columns=metric_cols))

display(result["feature_importance"].head(20))

for filename in [
    "lightgbm_roc_curve.png",
    "lightgbm_pr_curve.png",
    "lightgbm_confusion_matrix.png",
    "lightgbm_feature_importance.png",
]:
    display(Image(filename=str(PLOTS_DIR / filename)))


In [ ]:
final_model = train_final_model(X, y)
save_final_model(final_model, feature_cols, metrics=result["metrics"])
print("Saved final LightGBM serving model.")
